# LP3 Practical 9 – Email Spam Classification using KNN and SVM
**Aim:** Classify emails into normal (0) and spam (1) using K-Nearest Neighbors and Support Vector Machine, then compare performance.

**Dataset:** `emails.csv` (5,172 rows; 3,002 columns). The first column is an email identifier, the last column `Prediction` is the target, and the word-count columns are features.

## Cell 1 – Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)


## Cell 2 – Load and inspect dataset

In [ ]:
from pathlib import Path

def get_dataset_path(filename):
    candidates = [
        Path.cwd() / "Datasets" / filename,
        Path.cwd().parent / "Datasets" / filename
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Keep the Datasets folder beside the Notebooks folder."
    )

DATA_PATH = get_dataset_path("emails.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("Target column:", df.columns[-1])
print(df.iloc[:, -1].value_counts())


## Cell 3 – Separate features and target

In [ ]:
# The first column is only an email ID; do not use it for prediction.
X = df.drop(columns=["Email No.", "Prediction"], errors="ignore")
y = df["Prediction"]

# Ensure all feature columns are numeric.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(0)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


## Cell 4 – Split and scale the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)


## Cell 5 – Train KNN classifier

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

knn_predictions = knn_model.predict(X_test_scaled)
print("KNN model trained.")


## Cell 6 – Train SVM classifier

In [ ]:
# LinearSVC is a linear Support Vector Machine classifier.
# It is efficient for this dataset with thousands of word-count features.
svm_model = LinearSVC(random_state=42, max_iter=10000)
svm_model.fit(X_train_scaled, y_train)

svm_predictions = svm_model.predict(X_test_scaled)
print("SVM model trained.")


## Cell 7 – Evaluate both models

In [ ]:
def classification_metrics(actual, predicted):
    return {
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "F1 Score": f1_score(actual, predicted, zero_division=0)
    }

results = pd.DataFrame([
    {"Model": "KNN", **classification_metrics(y_test, knn_predictions)},
    {"Model": "SVM (LinearSVC)", **classification_metrics(y_test, svm_predictions)}
])

display(results)


## Cell 8 – Confusion matrices and classification reports

In [ ]:
print("KNN Confusion Matrix:")
print(confusion_matrix(y_test, knn_predictions))
print("\nKNN Classification Report:")
print(classification_report(y_test, knn_predictions, zero_division=0))

print("\nSVM Confusion Matrix:")
print(confusion_matrix(y_test, svm_predictions))
print("\nSVM Classification Report:")
print(classification_report(y_test, svm_predictions, zero_division=0))


## Sample output / observation
- `Prediction = 0` means Not Spam.
- `Prediction = 1` means Spam.
- The notebook compares accuracy, precision, recall and F1-score.
- It also prints a confusion matrix for each model.